# 05 · XAI: LIME explanations and their fidelity
Buổi 18. For 200 held-out (sentence, aspect) pairs, LIME perturbs the words around the aspect and fits a local linear surrogate to the ALBERT sentiment model.
**Fidelity** = 1 − kernel-weighted mean |surrogate − model| on 100 fresh perturbations (the same function the API uses). Target > 90%. About 25 minutes on a T4.

In [ ]:
#@title Setup — run once per session (Runtime > Change runtime type > T4 GPU)
REPO_URL = "https://github.com/YoungsterDumb/reviewlens"  # @param {type:"string"}
HF_NAMESPACE = "DatDuc"  # @param {type:"string"}  your Hugging Face username
import os, sys, subprocess
if not os.path.exists("/content/reviewlens"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/reviewlens"], check=True)
else:
    subprocess.run(["git", "-C", "/content/reviewlens", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/reviewlens/training/requirements.txt"], check=True)
sys.path.insert(0, "/content/reviewlens/training")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to T4 GPU")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a token with WRITE access (huggingface.co/settings/tokens)
METRICS_REPO = f"{HF_NAMESPACE}/reviewlens-metrics"

In [ ]:
from pathlib import Path
from reviewlens_train.data import load_absa, to_pairs
from reviewlens_train.evaluate import evaluate_xai
splits = load_absa(Path("/content/data"))
test_pairs = to_pairs(splits["test"])
xai = evaluate_xai(f"{HF_NAMESPACE}/reviewlens-atsc", test_pairs, n=200)
xai

In [ ]:
from reviewlens_train.hub import update_metrics
update_metrics(METRICS_REPO, xai)

In [ ]:
import matplotlib.pyplot as plt
from app.models.text_models import SentimentClassifier
from app.xai.explain import explain_text
clf = SentimentClassifier(f"{HF_NAMESPACE}/reviewlens-atsc")
e = explain_text(clf, "The screen is gorgeous but the battery barely lasts four hours.", "battery")
words = [w["word"] for w in e["weights"]][::-1]; vals = [w["weight"] for w in e["weights"]][::-1]
plt.barh(words, vals, color=["#E86A57" if v > 0 else "#8A94A6" for v in vals])
plt.title(f"Why '{e['label']}' for 'battery' (fidelity {e['fidelity']:.0%})"); plt.show()